# ML Assignment 2 – Credit Card Default Prediction

Trains 5 classifiers on the UCI *Default of Credit Card Clients* dataset, evaluates 6 metrics on a held-out test set, and saves the models for the Streamlit app.

## 0. Imports and settings

In [1]:
%pip install --upgrade numpy scipy scikit-learn pandas joblib

  Using cached numpy-2.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
Using cached numpy-2.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached scipy-1.18.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.1
    Uninstalling scipy-1.13.1:
      Successfully uninstalled scipy-1.13.1
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1

In [2]:
import os
import time

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
)

DATA_FILE = "/home/s2026ak05705/Desktop/credit_card.csv"
TARGET = "Y"
MODEL_DIR = "model"
RANDOM_STATE = 42

/home/s2026ak05705/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/home/s2026ak05705/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## 1. Load data

In [3]:
df = pd.read_csv(DATA_FILE)

# The first column is just the old row index saved to file - not a real feature
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{y.value_counts(normalize=True).round(3)}\n")

Dataset shape: (30000, 24)
Class balance:
Y
0    0.779
1    0.221
Name: proportion, dtype: float64



## 2. Train / test split (stratified because classes are imbalanced ~78/22)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Save test split (features + target) - this is what gets uploaded to the app
test_df = X_test.copy()
test_df[TARGET] = y_test
test_df.to_csv("test_data.csv", index=False)
print(f"Train: {X_train.shape}, Test: {X_test.shape}  -> test_data.csv saved\n")

Train: (24000, 23), Test: (6000, 23)  -> test_data.csv saved



## 3. Define models

Each model is a Pipeline(scaler -> classifier) so the saved file carries its own preprocessing, and the app only has to call .predict(). class_weight="balanced" makes models pay attention to the minority class.

In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=50, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(n_neighbors=15, weights="distance"),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(
        kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
        probability=True, random_state=RANDOM_STATE
    ),
}

# SVM with probability=True is slow on 24k rows; train it on a stratified sample
SVM_TRAIN_ROWS = 8000

## 4. Train, evaluate, save

In [6]:
os.makedirs(MODEL_DIR, exist_ok=True)
results = []

for name, clf in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])

    Xtr, ytr = X_train, y_train
    if name == "Support Vector Machine" and len(X_train) > SVM_TRAIN_ROWS:
        Xtr, _, ytr, _ = train_test_split(
            X_train, y_train, train_size=SVM_TRAIN_ROWS,
            stratify=y_train, random_state=RANDOM_STATE
        )

    start = time.time()
    pipe.fit(Xtr, ytr)
    train_time = time.time() - start

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

    file_name = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipe, os.path.join(MODEL_DIR, file_name))
    print(f"{name:<25} trained in {train_time:6.1f}s -> {file_name}")

Logistic Regression       trained in    1.2s -> logistic_regression.joblib
Decision Tree             trained in    0.3s -> decision_tree.joblib
KNN                       trained in    0.0s -> knn.joblib
Naive Bayes               trained in    0.0s -> naive_bayes.joblib


/home/s2026ak05705/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Support Vector Machine    trained in   34.8s -> support_vector_machine.joblib


## 5. Comparison table

In [7]:
metrics_df = pd.DataFrame(results).round(4)
metrics_df.to_csv(os.path.join(MODEL_DIR, "metrics.csv"), index=False)

print("\n=== Model comparison on test set ===")
print(metrics_df.to_string(index=False))
try:
    print("\nMarkdown table for README:\n")
    print(metrics_df.to_markdown(index=False))  # needs: pip install tabulate
except ImportError:
    pass


=== Model comparison on test set ===
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.6797 0.7081     0.3672  0.6202 0.4613 0.2718
         Decision Tree    0.7812 0.7651     0.5049  0.5403 0.5220 0.3807
                   KNN    0.8077 0.7302     0.6183  0.3406 0.4393 0.3564
           Naive Bayes    0.7525 0.7249     0.4515  0.5539 0.4975 0.3386
Support Vector Machine    0.7768 0.7487     0.4959  0.5509 0.5220 0.3778

Markdown table for README:

| ML Model Name          |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression    |     0.6797 | 0.7081 |      0.3672 |   0.6202 | 0.4613 | 0.2718 |
| Decision Tree          |     0.7812 | 0.7651 |      0.5049 |   0.5403 | 0.522  | 0.3807 |
| KNN                    |     0.8077 | 0.7302 |      0.6183 |   0.3406 | 0.4393 | 0.3564 |
| Naive Bayes            |     0.75